In [7]:
from pathlib import Path

import numpy as np
import torch
import os
import sys
import viser
import nerfview
from gsplat.rendering import rasterization

if not torch.cuda.is_available():
    raise RuntimeError("This demo requires CUDA for gsplat rasterization.")

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

device = torch.device("cuda")

inference_dir = Path(
    "/root/drivestudio-coding/outputs/minimal_sf/minimal_sf_stage4_3_multi_scene_v4_test/test/scene_001/segment_000/inference_only"
)
state_path = inference_dir / "3dgs_final.pt"  # 可改成 3dgs_init.pt

if not state_path.exists():
    raise FileNotFoundError(f"3DGS state not found: {state_path}")

viewer_host = "0.0.0.0"
viewer_port = 8098

max_points = 0  # 0 表示不裁剪点数
rng_seed = 42
initial_viewer_res = 512

In [8]:
state = torch.load(str(state_path), map_location="cpu")
branches = state.get("branches") or {}

parts_means = []
parts_scales = []
parts_quats = []
parts_opacities = []
parts_colors = []

# gsplat 的当前调用路径使用 colors=[N,3] 更稳；
# 因此这里用 sh_dc 还原 RGB，而不是直接传 SH 系数张量 [N,K,3]。
C0 = 0.28209479177387814

for branch_name in ("bg", "distant"):
    b = branches.get(branch_name)
    if not isinstance(b, dict):
        continue
    if b.get("means") is None:
        continue

    means_b = b["means"].float()
    scales_b = torch.exp(b["scales_log"].float())
    quats_b = b["quats"].float()
    opacities_b = torch.sigmoid(b["opacity_logit"].float()).squeeze(-1)
    sh_dc = b["sh_dc"].float()
    colors_b = torch.clamp(sh_dc * C0 + 0.5, 0.0, 1.0)  # [N,3]

    parts_means.append(means_b)
    parts_scales.append(scales_b)
    parts_quats.append(quats_b)
    parts_opacities.append(opacities_b)
    parts_colors.append(colors_b)

if len(parts_means) == 0:
    raise ValueError("No bg/distant branches found in 3DGS state.")

means = torch.cat(parts_means, dim=0)
scales = torch.cat(parts_scales, dim=0)
quats = torch.cat(parts_quats, dim=0)
opacities = torch.cat(parts_opacities, dim=0)
colors = torch.cat(parts_colors, dim=0)

num_total = int(means.shape[0])
if int(max_points) > 0 and num_total > int(max_points):
    rng = np.random.default_rng(int(rng_seed))
    keep = np.sort(rng.choice(num_total, size=int(max_points), replace=False))
    keep_t = torch.from_numpy(keep).long()
    means = means[keep_t]
    scales = scales[keep_t]
    quats = quats[keep_t]
    opacities = opacities[keep_t]
    colors = colors[keep_t]

means = means.to(device)
scales = scales.to(device)
quats = quats.to(device)
opacities = opacities.to(device)
colors = colors.to(device)

print(f"Loaded: {state_path}")
print("Branches used: bg + distant")
print(f"Num gaussians (raw bg+distant): {num_total}")
print(f"Num gaussians (used): {means.shape[0]}")
print(f"colors shape for rasterization: {tuple(colors.shape)}")

Loaded: /root/drivestudio-coding/outputs/minimal_sf/minimal_sf_stage4_3_multi_scene_v4_test/test/scene_001/segment_000/inference_only/3dgs_final.pt
Branches used: bg + distant
Num gaussians (raw bg+distant): 1200000
Num gaussians (used): 1200000
colors shape for rasterization: (1200000, 3)


In [ ]:
@torch.no_grad()
def render_fn(camera_state: nerfview.CameraState, img_wh):
    w, h = img_wh
    c2w = torch.from_numpy(camera_state.c2w).float().to(device)
    K = torch.from_numpy(camera_state.get_K(img_wh)).float().to(device)

    render_colors, _, _ = rasterization(
        means=means,
        quats=quats,
        scales=scales,
        opacities=opacities,
        colors=colors,
        viewmats=torch.linalg.inv(c2w)[None, ...],
        Ks=K[None, ...],
        width=int(w),
        height=int(h),
        packed=True,
        rasterize_mode="antialiased",
    )
    rgb = torch.clamp(render_colors[0], 0.0, 1.0).cpu().numpy()
    return (rgb * 255.0).astype(np.uint8)

# 重复运行时释放旧 server，避免端口/线程残留
if "_sf_demo_server" in globals() and _sf_demo_server is not None:
    try:
        _sf_demo_server.stop()
    except Exception:
        pass

_sf_demo_server = viser.ViserServer(host=viewer_host, port=int(viewer_port), verbose=False)
_sf_demo_viewer = nerfview.Viewer(
    server=_sf_demo_server,
    render_fn=render_fn,
    mode="rendering",
)

# 降低初始 viewer 分辨率，避免首次相机交互就爆显存
_sf_demo_viewer.render_tab_state.viewer_res = int(initial_viewer_res)
h_slider = _sf_demo_viewer._rendering_tab_handles.get("viewer_res_slider")
if h_slider is not None:
    h_slider.value = int(initial_viewer_res)

print(f"Viewer running at: http://localhost:{viewer_port}")
print("This demo visualizes only bg + distant style point cloud (no sky / no rigid).")
print(f"Viewer initial resolution: {initial_viewer_res}")

(viser) Server stopped

╭────── viser (listening *:8098) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8098   │
│   Websocket │ ws://localhost:8098     │
│             ╵                         │
╰───────────────────────────────────────╯

Viewer running at: http://localhost:8098
This demo visualizes only bg + distant style point cloud (no sky / no rigid).
Viewer initial resolution: 512


[WARNING] Your API will be deprecated in the future, please update your render_fn.


## 使用说明

1. 按顺序运行上面 3 个代码单元。
2. 打开输出的 URL（默认 `http://localhost:8098`）。
3. 默认读取 `3dgs_final.pt`，如果想看初始化状态，把 `state_path` 改成 `3dgs_init.pt`。
4. 该 notebook 只渲染 `bg + distant`，不会渲染 `sky` 和 `rigid`。
5. 当前默认不裁剪点数（`max_points=0`）。若你后续需要排查性能问题，再把 `max_points` 设成正数。